# Part 8: DAG-CBOR

DAG-CBOR is the deterministic encoding format used throughout ATProto.
It is a restricted profile of CBOR (RFC 8949) that guarantees a canonical
byte representation for any given data structure. This determinism is
essential for content addressing — the same logical data must always
produce the same bytes, and therefore the same CID.

**What you'll learn:**
- CBOR major types and encoding rules
- DAG-CBOR constraints (deterministic encoding)
- CBOR link type (tag 42) for content-addressed references
- Encoding/decoding round trips

**Prerequisites:** Part 7 (CID Content Addressing)

**Estimated Time:** 20-25 minutes

## Step 1: CBOR Major Types

CBOR encodes data into major types, each with a 3-bit type identifier
and a 5-bit additional info field:

| Major Type | Meaning | Example |
|------------|---------|----------|
| 0 | Unsigned integer | `0x00` = 0, `0x17` = 23 |
| 1 | Negative integer | `0x20` = -1, `0x29` = -10 |
| 2 | Byte string | `0x44` + 4 bytes |
| 3 | Text string | `0x64` + 4 UTF-8 bytes |
| 4 | Array | `0x82` = array of 2 items |
| 5 | Map | `0xa2` = map of 2 pairs |
| 6 | Tag | `0xd82a` = tag 42 (CID link) |
| 7 | Simple/float | `0xf4` = false, `0xf5` = true, `0xf6` = null |

In [ ]:
// CBOR encoder — simplified for demonstration
// Real implementation: Garazyk/Sources/Repository/CBOR.m

@interface CborEncoder : NSObject
- (NSData *)encodeInt:(int)value;
- (NSData *)encodeString:(NSString *)str;
@end

@implementation CborEncoder
- (NSData *)encodeInt:(int)value {
    // Major type 0: unsigned integer
    if (value < 24) {
        unsigned char byte = (0 << 5) | value;
        return [NSData dataWithBytes:&byte length:1];
    } else if (value < 256) {
        unsigned char b0 = 0x18;
        unsigned char b1 = (unsigned char)value;
        NSData *d0 = [NSData dataWithBytes:&b0 length:1];
        NSData *d1 = [NSData dataWithBytes:&b1 length:1];
        return [d0 dataByAppendingData:d1];
    }
    return [NSData data];
}
- (NSData *)encodeString:(NSString *)str {
    // Major type 3: text string
    int len = [str length];
    NSData *headerData;
    if (len < 24) {
        unsigned char header = (3 << 5) | len;
        headerData = [NSData dataWithBytes:&header length:1];
    } else {
        unsigned char headerBytes[2] = {(3 << 5) | 24, (unsigned char)len};
        headerData = [NSData dataWithBytes:headerBytes length:2];
    }
    const char *utf8 = [str UTF8String];
    NSData *strData = [NSData dataWithBytes:utf8 length:len];
    return [headerData dataByAppendingData:strData];
}
@end

CborEncoder *encoder = [[CborEncoder alloc] init];

// Encode some values
NSData *zero = [encoder encodeInt:0];
NSLog(@"CBOR 0: %d bytes", [zero length]);

NSData *hello = [encoder encodeString:@"hello"];
NSLog(@"CBOR 'hello': %d bytes", [hello length]);

NSData *ten = [encoder encodeInt:10];
NSLog(@"CBOR 10: %d bytes", [ten length]);

## Step 2: DAG-CBOR Constraints

DAG-CBOR adds these constraints on top of CBOR:

1. **Map keys must be text strings**, sorted by UTF-8 byte order
2. **Integers must use the shortest encoding** (no leading zeros)
3. **No indefinite-length items** (all lengths are known)
4. **No tags except tag 42** (CID link)
5. **No simple values except true, false, null**

These constraints guarantee that the same logical data always produces
the same bytes. This is why DAG-CBOR is used for content addressing.

In [ ]:
// Deterministic map encoding — keys sorted by byte order
@interface DAGCBORMap : NSObject
@property (nonatomic, strong) NSMutableDictionary *entries;
- (void)setValue:(id)value forKey:(NSString *)key;
- (NSArray *)sortedKeys;
@end

@implementation DAGCBORMap
- (instancetype)init {
    self = [super init];
    if (self) { _entries = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)setValue:(id)value forKey:(NSString *)key {
    [_entries setObject:value forKey:key];
}
- (NSArray *)sortedKeys {
    // DAG-CBOR requires keys sorted by UTF-8 byte order
    NSMutableArray *keys = [NSMutableArray array];
    for (NSString *k in _entries) { [keys addObject:k]; }
    [keys sortUsingSelector:@selector(compare:)];
    return keys;
}
@end

DAGCBORMap *map = [[DAGCBORMap alloc] init];
[map setValue:@"post" forKey:@"type"];
[map setValue:@"hello" forKey:@"text"];
[map setValue:@"did:plc:abc" forKey:@"did"];

NSArray *sorted = [map sortedKeys];
NSLog(@"Sorted keys: %@", sorted);
// DAG-CBOR encodes in this order: did, text, type

## Step 3: CBOR Link Type (Tag 42)

DAG-CBOR uses tag 42 to represent CID links. A link is a byte string
containing the raw binary CID (without multibase prefix), wrapped in
a tag 42 header. This is how ATProto records reference other blocks.

In [ ]:
// CBOR link — tag 42 wrapping a CID
@interface CborLink : NSObject
- (NSData *)encodeWithCID:(NSString *)cid;
@end

@implementation CborLink
- (NSData *)encodeWithCID:(NSString *)cid {
    // Tag 42 = 0xd82a in CBOR
    unsigned char tag0 = 0xd8;
    unsigned char tag1 = 0x2a;
    NSData *tagData = [NSData dataWithBytes:&tag0 length:1];
    tagData = [tagData dataByAppendingData:[NSData dataWithBytes:&tag1 length:1]];
    // Byte string header (simplified)
    int cidLen = [cid length];
    unsigned char bsHeader = (2 << 5) | cidLen;
    NSData *headerData = [NSData dataWithBytes:&bsHeader length:1];
    const char *cidBytes = [cid UTF8String];
    NSData *cidData = [NSData dataWithBytes:cidBytes length:cidLen];
    NSData *step1 = [tagData dataByAppendingData:headerData];
    return [step1 dataByAppendingData:cidData];
}
@end

In [ ]:
CborLink *link = [[CborLink alloc] init];
NSData *encoded = [link encodeWithCID:@"bafyrei12345678"];
NSLog(@"CBOR link: %d bytes", [encoded length]);
// First 2 bytes: 0xd8 0x2a (tag 42)
// Then: byte string with CID bytes

## Step 4: Garazyk Production Anchor

In the Garazyk codebase, `Repository/CBOR.h` and `Repository/CBOR.m` implement
full CBOR encoding/decoding:

- `CBORValue` — typed wrapper for all CBOR major types
- `CBOREncoder` — encodes `CBORValue` to bytes
- `CBORDecoder` — decodes bytes to `CBORValue`
- `CBORValue.tag:42` — represents a CID link

The production CBOR implementation handles all major types, indefinite-length
sequences, and proper UTF-8 key sorting. Our notebook uses simplified encoders
that demonstrate the concepts without full generality.

## Exercise

Add a `encodeBool:` method to `CborEncoder` that encodes a boolean value.
In CBOR, `true` is `0xf5` and `false` is `0xf4` (major type 7, simple values 21 and 20).

Hint: return a single-byte NSData with the appropriate CBOR simple value.

In [ ]:
// Exercise: implement encodeBool:
// Your code here...

## Solution

In [ ]:
// Solution: encodeBool produces CBOR simple values
@interface CborEncoder2 : NSObject
- (NSData *)encodeBool:(BOOL)value;
@end

@implementation CborEncoder2
- (NSData *)encodeBool:(BOOL)value {
    // Major type 7: simple values
    // false = 0xf4 (7<<5 | 20), true = 0xf5 (7<<5 | 21)
    unsigned char byte = value ? 0xf5 : 0xf4;
    return [NSData dataWithBytes:&byte length:1];
}
@end

CborEncoder2 *enc = [[CborEncoder2 alloc] init];
NSData *t = [enc encodeBool:YES];
NSData *f = [enc encodeBool:NO];
NSLog(@"true: %d byte(s)", [t length]);
NSLog(@"false: %d byte(s)", [f length]);

## What to Read Next

You now understand DAG-CBOR encoding. Next:
- **Part 9: CAR Archives** — how CBOR blocks are packaged into CAR files
- **Part 10: MST Insertion** — how MST nodes are serialized as DAG-CBOR
- **Part 14: Record Writes** — how records are CBOR-encoded before storage